# Data Saturation Analysis with Concrete Autoencoder

## Overview
This notebook performs a comprehensive data saturation analysis to determine the optimal training set size for a Concrete Autoencoder (CAE) model. The analysis evaluates how model performance changes as we vary the fraction of available training data from 10% to 100%.

## Objectives
- **Evaluate Training Data Requirements**: Test CAE performance across different sample sizes (0.1 to 1.0 in 0.1 increments)
- **Identify Saturation Point**: Determine the minimum training data size needed to achieve optimal performance
- **Performance Metrics**: Track validation loss, test loss, feature reconstruction error, and critical wavelength reconstruction quality
- **Model Persistence**: Save trained models and performance metrics to S3 for further analysis

## Methodology
The notebook uses a Concrete Autoencoder with Gumbel-softmax feature selection to:
1. Compress spectral data from 277+ features to 32 selected features
2. Reconstruct the original input through a decoder network
3. Evaluate reconstruction quality across different training set sizes
4. Focus on critical wavelengths (630-650nm range) for specialized analysis

## Setup
Run all cells sequentially. Make sure you have a `.env` file with your AWS configuration!

In [ ]:
import os

import shutil
import boto3

import pandas as pd
from dotenv import load_dotenv
import pyarrow.dataset as ds
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import numpy as np

load_dotenv()

True

In [ ]:
class ConcreteSelect(layers.Layer):
    def __init__(self, k, input_dim, temperature=0.1, **kwargs):
        super().__init__(**kwargs)
        self.k = k
        self.input_dim = input_dim
        self.temperature = temperature

    def build(self, input_shape):
        self.logits = self.add_weight(
            shape=(self.k, self.input_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='logits'
        )

    def call(self, inputs, training=None):
        if training:
            uniform = tf.random.uniform(tf.shape(self.logits), minval=0, maxval=1)
            gumbel = -tf.math.log(-tf.math.log(uniform + 1e-20) + 1e-20)
            noisy_logits = (self.logits + gumbel) / self.temperature
            scores = tf.nn.softmax(noisy_logits, axis=-1)
        else:
            scores = tf.one_hot(tf.argmax(self.logits, axis=-1), depth=self.input_dim)
        return tf.matmul(inputs, tf.transpose(scores))

    def get_config(self):
        config = super().get_config()
        config.update({
            "k": self.k,
            "input_dim": self.input_dim,
            "temperature": self.temperature
        })
        return config

def get_results(hist, autoencoder, percentage):
  # Validation loss
  val_loss = hist.history["val_loss"][-1]

  # Test loss
  X_reconstructed = autoencoder.predict(X_test.values)
  test_loss = float(np.mean(np.square(X_test.values - X_reconstructed)))

  # Per feature
  errors = np.square(X_test - X_reconstructed)
  feature_mse = np.mean(errors, axis=0)
  features_95_percentile = np.percentile(feature_mse, 95)

  # Per pixel
  errors = np.square(X_test - X_reconstructed)
  pixel_mse = np.mean(errors, axis=1)
  pixel_95_percentile = np.percentile(pixel_mse, 95)

  # Per Critical Wavelengths
  critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
  critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
  critical_wavelength_mse = feature_mse[critical_wavelengths]
  critical_99_percentile = np.percentile(critical_wavelength_mse, 99)


  performance = {
      "percentage": percentage,
      "val_loss": val_loss,
      "test_loss": test_loss,
      "features_95_percentile": features_95_percentile,
      "pixel_95_percentile": pixel_95_percentile,
      "critical_wavelength_99_percentile": critical_99_percentile
  }

  return pd.Series(performance)

# 1. Import Data

In [ ]:
bucket = os.getenv("BUCKET_NAME")
results_bucket = os.getenv("SATURATION_BUCKET_NAME")

column_path = f"s3://{bucket}/headers.parquet"
columns =  pd.read_parquet(column_path)['0'].values

s3_folder = f"s3://{bucket}/samples/"
dataset = ds.dataset(s3_folder, format="parquet")
toscore = dataset.to_table().to_pandas()

toscore.columns = columns

bad_wavelengths = [c for c in toscore.columns if c < 320 or (c > 590 and c < 610)]

X = toscore.drop(columns=bad_wavelengths).dropna()

X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

array([314.5071 , 316.10037, 318.14838, 320.16986, 322.307  , 324.50623,
       326.71512, 328.91248, 331.29648, 334.04498], dtype=float32)

# 2. Data Saturation Check

In [ ]:
seed = 42
sample_sizes = np.arange(0.1, 1.1, 0.1)
num_features = 277
k = 32
version = "v1"

for sample_size in sample_sizes:
  print(f"Sample Size: {sample_size}")
  print("="*60)
  X_sample = X.sample(frac=sample_size, random_state=seed)
  X_train, X_test = train_test_split(X_sample, test_size=0.2, random_state=seed)
  X_scaled = X_train.dropna().values

  input_dim = X_scaled.shape[1]

  input_layer = keras.Input(shape=(input_dim,))
  encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

  decoded = layers.Dense(64, activation="relu")(encoded)
  decoded = layers.Dense(128, activation="relu")(encoded)
  decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

  cae = keras.Model(inputs=input_layer, outputs=decoded)
  cae.compile(optimizer="adam", loss="mse")

  cae_hist = cae.fit(
      X_scaled, X_scaled,
      epochs=50,
      batch_size=32,
      shuffle=True,
      validation_split=0.2,
      verbose=0
  )

  encoder = keras.Model(inputs=input_layer, outputs=encoded)
  X_encoded = encoder.predict(X_scaled)

  X_reconstructed = cae.predict(X_scaled)

  performance = get_results(cae_hist, cae, sample_size).to_frame()

  validation_path = f"s3://{results_bucket}/{sample_size}/{version}/score.parquet"
  weights_path = f"{sample_size}/{version}/cae_savedmodel.zip"

  performance.to_parquet(
        validation_path,
        index=True,
        engine="pyarrow",
    )

  os.makedirs("cae_artifacts", exist_ok=True)
  cae.save_weights("cae_artifacts/cae_weights.weights.h5")

  zip_path = shutil.make_archive("cae_savedmodel", "zip", "cae_artifacts")
  s3 = boto3.client("s3")
  s3.upload_file(zip_path, results_bucket, weights_path)

  print(performance)